In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import xgboost as xgb
from sklearn.neural_network import MLPClassifier

# For saving models
import pickle
import joblib

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load the dataset
df = pd.read_csv('telecom_customer_churn.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\n" + "="*50)
print("Dataset Info:")
print("="*50)
df.info()
print("\n" + "="*50)
print("First 5 rows:")
print("="*50)
df.head()

Dataset Shape: (7043, 38)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 38 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Number of Referrals                7043 non-null   int64  
 10  Tenure in Months                   7043 non-null   int64  
 11  Offer          

,Customer ID,Gender,Age,Married,Number of Dependents,City,Zip Code,Latitude,Longitude,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Customer Status,Churn Category,Churn Reason
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.83,-119.00,2,9,NaN,Yes,42.39,No,Yes,Cable,16.00,No,Yes,No,Yes,Yes,No,No,Yes,One Year,Yes,Credit Card,65.60,593.30,0.00,0,381.51,974.81,Stayed,NaN,NaN
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.16,-118.20,0,9,NaN,Yes,10.69,Yes,Yes,Cable,10.00,No,No,No,No,No,Yes,Yes,No,Month-to-Month,No,Credit Card,-4.00,542.40,38.33,10,96.21,610.28,Stayed,NaN,NaN
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.65,-117.92,0,4,Offer E,Yes,33.65,No,Yes,Fiber Optic,30.00,No,No,Yes,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,73.90,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices
3,0011-IGKFF,Male,78,Yes,0,Martinez,94553,38.01,-122.12,1,13,Offer D,Yes,27.82,No,Yes,Fiber Optic,4.00,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.00,1237.85,0.00,0,361.66,1599.51,Churned,Dissatisfaction,Product dissatisfaction
4,0013-EXCHZ,Female,75,Yes,0,Camarillo,93010,34.23,-119.08,3,3,NaN,Yes,7.38,No,Yes,Fiber Optic,11.00,No,No,No,Yes,Yes,No,No,Yes,Month-to-Month,Yes,Credit Card,83.90,267.40,0.00,0,22.14,289.54,Churned,Dissatisfaction,Network reliability


In [3]:
# Display all columns and their data types
print("Column Names and Types:")
print("="*50)
for col, dtype in zip(df.columns, df.dtypes):
    print(f"{col}: {dtype}")

# Check for unique values in each column
print("\n" + "="*50)
print("Unique Values Count for Each Column:")
print("="*50)
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

Column Names and Types:
Customer ID: object
Gender: object
Age: int64
Married: object
Number of Dependents: int64
City: object
Zip Code: int64
Latitude: float64
Longitude: float64
Number of Referrals: int64
Tenure in Months: int64
Offer: object
Phone Service: object
Avg Monthly Long Distance Charges: float64
Multiple Lines: object
Internet Service: object
Internet Type: object
Avg Monthly GB Download: float64
Online Security: object
Online Backup: object
Device Protection Plan: object
Premium Tech Support: object
Streaming TV: object
Streaming Movies: object
Streaming Music: object
Unlimited Data: object
Contract: object
Paperless Billing: object
Payment Method: object
Monthly Charge: float64
Total Charges: float64
Total Refunds: float64
Total Extra Data Charges: int64
Total Long Distance Charges: float64
Total Revenue: float64
Customer Status: object
Churn Category: object
Churn Reason: object

Unique Values Count for Each Column:
Customer ID: 7043 unique values
Gender: 2 unique value

In [4]:
# Check for missing values
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)

if len(missing_data) > 0:
    print("Missing Values Summary:")
    print("="*50)
    print(missing_data)
    
    # Visualize missing values
    fig = px.bar(missing_data, x='Column', y='Missing_Percentage',
                 title='Missing Values Percentage by Column',
                 labels={'Missing_Percentage': 'Percentage (%)'},
                 color='Missing_Percentage',
                 color_continuous_scale='Reds')
    fig.update_layout(height=400)
    fig.show()
else:
    print("No missing values found in the dataset!")

Missing Values Summary:
                                                              Column  \
Churn Category                                        Churn Category   
Churn Reason                                            Churn Reason   
Offer                                                          Offer   
Online Security                                      Online Security   
Online Backup                                          Online Backup   
Avg Monthly GB Download                      Avg Monthly GB Download   
Internet Type                                          Internet Type   
Streaming Movies                                    Streaming Movies   
Streaming TV                                            Streaming TV   
Device Protection Plan                        Device Protection Plan   
Premium Tech Support                            Premium Tech Support   
Unlimited Data                                        Unlimited Data   
Streaming Music                         

In [5]:
# Handle missing values based on column type
def handle_missing_values(df):
    df_processed = df.copy()
    
    # For numeric columns, fill with median or mean
    numeric_columns = df_processed.select_dtypes(include=['float64', 'int64']).columns
    for col in numeric_columns:
        if df_processed[col].isnull().sum() > 0:
            # Use median for skewed distributions
            df_processed[col].fillna(df_processed[col].median(), inplace=True)
    
    # For categorical columns, fill with mode or 'Unknown'
    categorical_columns = df_processed.select_dtypes(include=['object']).columns
    for col in categorical_columns:
        if df_processed[col].isnull().sum() > 0:
            # If it's a service-related column and customer doesn't have the base service
            if 'No internet service' in df_processed[col].values or 'No phone service' in df_processed[col].values:
                df_processed[col].fillna('No service', inplace=True)
            else:
                df_processed[col].fillna(df_processed[col].mode()[0] if len(df_processed[col].mode()) > 0 else 'Unknown', inplace=True)
    
    return df_processed

df_clean = handle_missing_values(df)
print("Missing values handled successfully!")
print(f"Remaining missing values: {df_clean.isnull().sum().sum()}")

Missing values handled successfully!
Remaining missing values: 0


In [6]:
# Identify the target variable (commonly 'Churn' or similar)
target_columns = [col for col in df_clean.columns if 'churn' in col.lower()]
print("Potential target columns:", target_columns)

# Assuming 'Churn' is the target variable (adjust if different)
if 'Churn' in df_clean.columns:
    target_col = 'Churn'
elif 'Customer Status' in df_clean.columns:
    target_col = 'Customer Status'
    # Create binary churn column
    df_clean['Churn'] = df_clean[target_col].apply(lambda x: 1 if 'Churned' in str(x) else 0)
    target_col = 'Churn'
else:
    # Look for any column that might indicate churn
    for col in target_columns:
        if col in df_clean.columns:
            target_col = col
            break

print(f"Target variable: {target_col}")
print("\nTarget distribution:")
print(df_clean[target_col].value_counts())
print("\nTarget distribution (%):")
print(df_clean[target_col].value_counts(normalize=True) * 100)

# Visualize target distribution
fig = px.pie(values=df_clean[target_col].value_counts().values, 
             names=df_clean[target_col].value_counts().index,
             title='Target Variable Distribution (Churn)',
             color_discrete_sequence=['#00CC96', '#EF553B'])
fig.show()

Potential target columns: ['Churn Category', 'Churn Reason']
Target variable: Churn

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Target distribution (%):
Churn
0   73.46
1   26.54
Name: proportion, dtype: float64


In [7]:
# Separate numerical and categorical columns
numerical_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()

# Remove target from numerical columns if present
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols[:5]}...")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols[:5]}...")

# Statistical summary of numerical features
print("\n" + "="*50)
print("Numerical Features Statistics:")
print("="*50)
df_clean[numerical_cols].describe()

Numerical columns (15): ['Age', 'Number of Dependents', 'Zip Code', 'Latitude', 'Longitude']...
Categorical columns (23): ['Customer ID', 'Gender', 'Married', 'City', 'Offer']...

Numerical Features Statistics:


,Age,Number of Dependents,Zip Code,Latitude,Longitude,Number of Referrals,Tenure in Months,Avg Monthly Long Distance Charges,Avg Monthly GB Download,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue
count,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00,7043.00
mean,46.51,0.47,93486.07,36.20,-119.76,1.95,32.39,25.45,25.07,63.60,2280.38,1.96,6.86,749.10,3034.38
std,16.75,0.96,1856.77,2.47,2.15,3.00,24.54,13.50,17.47,31.20,2266.22,7.90,25.10,846.66,2865.20
min,19.00,0.00,90001.00,32.56,-124.30,0.00,1.00,1.01,2.00,-10.00,18.80,0.00,0.00,0.00,21.36
25%,32.00,0.00,92101.00,33.99,-121.79,0.00,9.00,14.46,15.00,30.40,400.15,0.00,0.00,70.55,605.61
50%,46.00,0.00,93518.00,36.21,-119.60,0.00,29.00,25.69,21.00,70.05,1394.55,0.00,0.00,401.44,2108.64
75%,60.00,0.00,95329.00,38.16,-117.97,3.00,55.00,36.39,27.00,89.75,3786.60,0.00,0.00,1191.10,4801.15
max,80.00,9.00,96150.00,41.96,-114.19,11.00,72.00,49.99,85.00,118.75,8684.80,49.79,150.00,3564.72,11979.34


In [8]:
# Create distribution plots for numerical features
n_cols = 3
n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

fig = make_subplots(rows=n_rows, cols=n_cols, 
                    subplot_titles=numerical_cols[:n_rows*n_cols],
                    vertical_spacing=0.1)

for idx, col in enumerate(numerical_cols[:n_rows*n_cols]):
    row = idx // n_cols + 1
    col_pos = idx % n_cols + 1
    
    if idx < len(numerical_cols):
        fig.add_trace(
            go.Histogram(x=df_clean[numerical_cols[idx]], 
                        name=numerical_cols[idx],
                        showlegend=False),
            row=row, col=col_pos
        )

fig.update_layout(height=300*n_rows, 
                  title_text="Distribution of Numerical Features",
                  showlegend=False)
fig.show()

In [9]:
# Calculate correlation matrix for numerical features
corr_matrix = df_clean[numerical_cols + [target_col]].corr()

# Create correlation heatmap
fig = px.imshow(corr_matrix, 
                text_auto=True,
                aspect="auto",
                color_continuous_scale='RdBu_r',
                title='Correlation Matrix of Numerical Features')
fig.update_layout(height=800, width=900)
fig.show()

# Find top correlations with target
target_corr = corr_matrix[target_col].sort_values(ascending=False)[1:]
print("Top 10 Features Correlated with Churn:")
print(target_corr.head(10))

Top 10 Features Correlated with Churn:
Monthly Charge                       0.19
Age                                  0.12
Longitude                            0.02
Total Extra Data Charges             0.01
Avg Monthly Long Distance Charges    0.00
Zip Code                            -0.02
Total Refunds                       -0.03
Latitude                            -0.04
Avg Monthly GB Download             -0.06
Total Charges                       -0.20
Name: Churn, dtype: float64


In [10]:
# Analyze categorical features vs target
fig_list = []
for cat_col in categorical_cols[:10]:  # Analyze top 10 categorical columns
    if cat_col != target_col:
        # Calculate churn rate for each category
        churn_rate = df_clean.groupby(cat_col)[target_col].mean().sort_values(ascending=False)
        
        fig = px.bar(x=churn_rate.index, y=churn_rate.values,
                     title=f'Churn Rate by {cat_col}',
                     labels={'x': cat_col, 'y': 'Churn Rate'},
                     color=churn_rate.values,
                     color_continuous_scale='Reds')
        fig.update_layout(height=400, showlegend=False)
        fig.show()

In [13]:
# Customer segmentation analysis
if 'TotalCharges' in df_clean.columns and 'tenure' in df_clean.columns:
    # Create customer value segments
    df_clean['CustomerValue'] = pd.qcut(df_clean['TotalCharges'], 
                                         q=4, 
                                         labels=['Low', 'Medium', 'High', 'Premium'])
    
    # Analyze churn by customer value
    churn_by_value = df_clean.groupby('CustomerValue')[target_col].agg(['mean', 'count'])
    
    fig = px.bar(churn_by_value, y='mean',
                 title='Churn Rate by Customer Value Segment',
                 labels={'mean': 'Churn Rate', 'index': 'Customer Segment'},
                 color='mean',
                 color_continuous_scale='RdYlGn_r',
                 text='mean')
    fig.update_traces(texttemplate='%{text:.2%}')
    fig.show()

In [14]:
# Create new features
df_processed = df_clean.copy()

# Example feature engineering
if 'MonthlyCharges' in df_processed.columns and 'TotalCharges' in df_processed.columns:
    df_processed['AvgChargesPerMonth'] = df_processed['TotalCharges'] / (df_processed['tenure'] + 1)
    df_processed['ChargesRatio'] = df_processed['MonthlyCharges'] / (df_processed['AvgChargesPerMonth'] + 0.1)

# Create tenure groups
if 'tenure' in df_processed.columns:
    df_processed['TenureGroup'] = pd.cut(df_processed['tenure'], 
                                         bins=[0, 12, 24, 36, 48, 60, 100],
                                         labels=['0-1yr', '1-2yr', '2-3yr', '3-4yr', '4-5yr', '5+yr'])

# Count number of services
service_cols = [col for col in df_processed.columns if any(word in col.lower() for word in ['service', 'streaming', 'backup', 'security'])]
if service_cols:
    df_processed['TotalServices'] = (df_processed[service_cols] == 'Yes').sum(axis=1)

print("Feature engineering completed!")
print(f"New features created: {set(df_processed.columns) - set(df_clean.columns)}")

Feature engineering completed!
New features created: {'TotalServices'}


In [15]:
# Encode categorical variables
le_dict = {}
df_encoded = df_processed.copy()

# Binary encoding for Yes/No columns
binary_cols = []
for col in df_encoded.columns:
    if df_encoded[col].dtype == 'object':
        unique_values = df_encoded[col].unique()
        if len(unique_values) == 2 or set(unique_values) <= {'Yes', 'No', 'Male', 'Female', 'True', 'False'}:
            binary_cols.append(col)
            le = LabelEncoder()
            df_encoded[col] = le.fit_transform(df_encoded[col].fillna('Unknown'))
            le_dict[col] = le

# One-hot encoding for multi-category columns
multi_cat_cols = [col for col in df_encoded.select_dtypes(include=['object']).columns if col not in binary_cols and col != target_col]
if multi_cat_cols:
    df_encoded = pd.get_dummies(df_encoded, columns=multi_cat_cols, prefix=multi_cat_cols, drop_first=True)

print(f"Binary encoded columns: {binary_cols}")
print(f"One-hot encoded columns: {multi_cat_cols}")
print(f"Final dataset shape: {df_encoded.shape}")

Binary encoded columns: ['Gender', 'Married', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Paperless Billing']
One-hot encoded columns: ['Customer ID', 'City', 'Offer', 'Internet Type', 'Contract', 'Payment Method', 'Customer Status', 'Churn Category', 'Churn Reason']
Final dataset shape: (7043, 8213)


In [16]:
# Prepare data for modeling
X = df_encoded.drop([target_col], axis=1)
y = df_encoded[target_col]

# Remove any non-numeric columns that might remain
X = X.select_dtypes(include=[np.number])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, 'scaler.pkl')

print(f"Training set size: {X_train_scaled.shape}")
print(f"Test set size: {X_test_scaled.shape}")
print(f"Class distribution in training set:\n{y_train.value_counts(normalize=True)}")

Training set size: (5634, 30)
Test set size: (1409, 30)
Class distribution in training set:
Churn
0   0.73
1   0.27
Name: proportion, dtype: float64


In [17]:
# Define models to train
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100),
    'XGBoost': xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    'AdaBoost': AdaBoostClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Neural Network': MLPClassifier(random_state=42, max_iter=1000)
}

# Train baseline models
baseline_results = {}

for name, model in models.items():
    print(f"Training {name}...")
    
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else 0
    
    baseline_results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Model': model
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print()

# Convert to DataFrame for better visualization
results_df = pd.DataFrame(baseline_results).T
results_df = results_df.drop('Model', axis=1)
results_df = results_df.sort_values('ROC-AUC', ascending=False)
print("\nBaseline Model Performance:")
print(results_df)

Training Logistic Regression...
  Accuracy: 0.8055
  ROC-AUC: 0.8686

Training Decision Tree...
  Accuracy: 0.7480
  ROC-AUC: 0.6902

Training Random Forest...
  Accuracy: 0.8176
  ROC-AUC: 0.8659

Training Gradient Boosting...
  Accuracy: 0.8282
  ROC-AUC: 0.8893

Training XGBoost...
  Accuracy: 0.8169
  ROC-AUC: 0.8754

Training AdaBoost...
  Accuracy: 0.8240
  ROC-AUC: 0.8772

Training SVM...
  Accuracy: 0.8112
  ROC-AUC: 0.8631

Training KNN...
  Accuracy: 0.7708
  ROC-AUC: 0.7818

Training Naive Bayes...
  Accuracy: 0.7495
  ROC-AUC: 0.8246

Training Neural Network...
  Accuracy: 0.8048
  ROC-AUC: 0.8414


Baseline Model Performance:
                    Accuracy Precision Recall F1-Score ROC-AUC
Gradient Boosting       0.83      0.70   0.62     0.66    0.89
AdaBoost                0.82      0.71   0.56     0.63    0.88
XGBoost                 0.82      0.67   0.61     0.64    0.88
Logistic Regression     0.81      0.65   0.59     0.62    0.87
Random Forest           0.82      0.70

In [18]:
# Create comparison visualization
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=('Accuracy Comparison', 'Precision Comparison',
                                  'Recall Comparison', 'ROC-AUC Comparison'))

metrics = ['Accuracy', 'Precision', 'Recall', 'ROC-AUC']
colors = px.colors.qualitative.Set3

for idx, metric in enumerate(metrics):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    values = results_df[metric].values
    models = results_df.index
    
    fig.add_trace(
        go.Bar(x=models, y=values, name=metric,
               marker_color=colors[idx],
               text=values,
               texttemplate='%{text:.3f}',
               showlegend=False),
        row=row, col=col
    )

fig.update_layout(height=700, title_text="Model Performance Comparison")
fig.show()

In [19]:
# Random Forest Hyperparameter Tuning
print("Performing Hyperparameter Tuning for Random Forest...")

rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2'],
    'bootstrap': [True, False]
}

# Use RandomizedSearchCV for faster results
rf_random = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=50,
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    scoring='roc_auc'
)

rf_random.fit(X_train_scaled, y_train)

print(f"Best parameters: {rf_random.best_params_}")
print(f"Best cross-validation score: {rf_random.best_score_:.4f}")

# Evaluate on test set
y_pred_rf = rf_random.predict(X_test_scaled)
y_pred_proba_rf = rf_random.predict_proba(X_test_scaled)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

Performing Hyperparameter Tuning for Random Forest...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': False}
Best cross-validation score: 0.8794
Test ROC-AUC: 0.8733


In [20]:
# XGBoost Hyperparameter Tuning
print("Performing Hyperparameter Tuning for XGBoost...")

xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.5, 1]
}

# Use RandomizedSearchCV
xgb_random = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    param_distributions=xgb_param_grid,
    n_iter=50,
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    scoring='roc_auc'
)

xgb_random.fit(X_train_scaled, y_train)

print(f"Best parameters: {xgb_random.best_params_}")
print(f"Best cross-validation score: {xgb_random.best_score_:.4f}")

# Evaluate on test set
y_pred_xgb = xgb_random.predict(X_test_scaled)
y_pred_proba_xgb = xgb_random.predict_proba(X_test_scaled)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")

Performing Hyperparameter Tuning for XGBoost...
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 0.8}
Best cross-validation score: 0.8930
Test ROC-AUC: 0.8874


In [ ]:
# Gradient Boosting Hyperparameter Tuning
print("Performing Grid Search for Gradient Boosting...")

gb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.8, 1.0]
}

gb_grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=gb_param_grid,
    cv=5,
    verbose=1,
    n_jobs=-1,
    scoring='roc_auc'
)

gb_grid.fit(X_train_scaled, y_train)

print(f"Best parameters: {gb_grid.best_params_}")
print(f"Best cross-validation score: {gb_grid.best_score_:.4f}")

# Evaluate on test set
y_pred_gb = gb_grid.predict(X_test_scaled)
y_pred_proba_gb = gb_grid.predict_proba(X_test_scaled)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba_gb):.4f}")

Performing Grid Search for Gradient Boosting...
Fitting 5 folds for each of 324 candidates, totalling 1620 fits


In [ ]:
# Perform cross-validation for top models
best_models = {
    'Random Forest (Tuned)': rf_random.best_estimator_,
    'XGBoost (Tuned)': xgb_random.best_estimator_,
    'Gradient Boosting (Tuned)': gb_grid.best_estimator_
}

cv_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in best_models.items():
    print(f"\nCross-validation for {name}:")
    
    # Calculate various CV scores
    cv_accuracy = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='accuracy')
    cv_roc_auc = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='roc_auc')
    cv_f1 = cross_val_score(model, X_train_scaled, y_train, cv=skf, scoring='f1')
    
    cv_results[name] = {
        'Accuracy': f"{cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})",
        'ROC-AUC': f"{cv_roc_auc.mean():.4f} (+/- {cv_roc_auc.std():.4f})",
        'F1-Score': f"{cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})"
    }
    
    print(f"  Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
    print(f"  ROC-AUC: {cv_roc_auc.mean():.4f} (+/- {cv_roc_auc.std():.4f})")
    print(f"  F1-Score: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")

# Display CV results
cv_results_df = pd.DataFrame(cv_results).T
print("\n" + "="*50)
print("Cross-Validation Results Summary:")
print("="*50)
print(cv_results_df)

In [ ]:
# Plot ROC curves for best models
fig = go.Figure()

best_models_for_roc = {
    'Random Forest': rf_random.best_estimator_,
    'XGBoost': xgb_random.best_estimator_,
    'Gradient Boosting': gb_grid.best_estimator_,
    'Logistic Regression': baseline_results['Logistic Regression']['Model']
}

colors = ['blue', 'green', 'red', 'purple']

for (name, model), color in zip(best_models_for_roc.items(), colors):
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode='lines',
        name=f'{name} (AUC = {auc:.3f})',
        line=dict(color=color, width=2)
    ))

# Add diagonal line
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random Classifier',
    line=dict(color='gray', width=1, dash='dash')
))

fig.update_layout(
    title='ROC Curves Comparison',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    height=500,
    width=700,
    showlegend=True,
    legend=dict(x=0.6, y=0.1)
)
fig.show()

In [ ]:
# Create confusion matrices for best models
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=list(best_models_for_roc.keys()),
                    vertical_spacing=0.15,
                    horizontal_spacing=0.15)

for idx, (name, model) in enumerate(best_models_for_roc.items()):
    row = idx // 2 + 1
    col = idx % 2 + 1
    
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    fig.add_trace(
        go.Heatmap(z=cm,
                   x=['Predicted No Churn', 'Predicted Churn'],
                   y=['Actual No Churn', 'Actual Churn'],
                   text=cm,
                   texttemplate='%{text}',
                   colorscale='Blues',
                   showscale=False),
        row=row, col=col
    )

fig.update_layout(height=700, title_text="Confusion Matrices for Best Models")
fig.show()

In [ ]:
# Get feature importance from tree-based models
feature_importance_dict = {}

# Random Forest
if hasattr(rf_random.best_estimator_, 'feature_importances_'):
    feature_importance_dict['Random Forest'] = pd.Series(
        rf_random.best_estimator_.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False).head(15)

# XGBoost
if hasattr(xgb_random.best_estimator_, 'feature_importances_'):
    feature_importance_dict['XGBoost'] = pd.Series(
        xgb_random.best_estimator_.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False).head(15)

# Gradient Boosting
if hasattr(gb_grid.best_estimator_, 'feature_importances_'):
    feature_importance_dict['Gradient Boosting'] = pd.Series(
        gb_grid.best_estimator_.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False).head(15)

# Visualize feature importance
for name, importance in feature_importance_dict.items():
    fig = px.bar(x=importance.values, y=importance.index,
                 orientation='h',
                 title=f'Top 15 Feature Importance - {name}',
                 labels={'x': 'Importance', 'y': 'Features'},
                 color=importance.values,
                 color_continuous_scale='Viridis')
    fig.update_layout(height=500)
    fig.show()

In [ ]:
# Create final evaluation summary
final_models = {
    'Random Forest': rf_random.best_estimator_,
    'XGBoost': xgb_random.best_estimator_,
    'Gradient Boosting': gb_grid.best_estimator_
}

final_results = []

for name, model in final_models.items():
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    final_results.append(results)
    
    print(f"\n{name} - Detailed Classification Report:")
    print("="*50)
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Create final comparison dataframe
final_results_df = pd.DataFrame(final_results)
final_results_df = final_results_df.set_index('Model')

print("\n" + "="*50)
print("FINAL MODEL COMPARISON:")
print("="*50)
print(final_results_df)

# Find best model
best_model_name = final_results_df['ROC-AUC'].idxmax()
print(f"\n🏆 Best Model: {best_model_name} with ROC-AUC of {final_results_df.loc[best_model_name, 'ROC-AUC']:.4f}")